# olist-v2 — cek stack

Memeriksa tiap layanan dari dalam container Jupyter. Setiap sel mencetak `OK` atau penyebab gagalnya, jadi bisa dijalankan walau sebagian profil belum menyala (Kafka butuh profil `stream`, Spark butuh profil `spark`).

In [ ]:
import os, sys, platform
print(platform.python_version(), '|', 'JAVA_HOME =', os.environ.get('JAVA_HOME'))
for k in ('MINIO_ENDPOINT','KAFKA_BOOTSTRAP','SPARK_MASTER','SPARK_CONF_DIR'):
    print(f'{k:16s}', os.environ.get(k))

## MinIO (S3)

In [ ]:
import os, boto3
try:
    s3 = boto3.client('s3', endpoint_url=os.environ['MINIO_ENDPOINT'],
                      aws_access_key_id=os.environ['MINIO_ROOT_USER'], aws_secret_access_key=os.environ['MINIO_ROOT_PASSWORD'],
                      region_name='us-east-1')
    print('OK bucket:', [b['Name'] for b in s3.list_buckets()['Buckets']])
except Exception as e:
    print('GAGAL MinIO:', type(e).__name__, str(e)[:160])

## Postgres (warehouse `olist_dw`)

In [ ]:
import os, psycopg
try:
    with psycopg.connect(os.environ['WAREHOUSE_DSN'], connect_timeout=5) as c:
        schemas = [r[0] for r in c.execute("select schema_name from information_schema.schemata where schema_name in ('raw','staging','dw','mart') order by 1")]
    print('OK schema:', schemas)
except Exception as e:
    print('GAGAL Postgres:', type(e).__name__, str(e)[:160])

## Kafka (profil `stream`)

In [ ]:
import os
try:
    from kafka.admin import KafkaAdminClient
    a = KafkaAdminClient(bootstrap_servers=os.environ['KAFKA_BOOTSTRAP'], request_timeout_ms=5000)
    print('OK topic:', sorted(a.list_topics()))
    a.close()
except Exception as e:
    print('GAGAL Kafka (profil stream menyala?):', type(e).__name__, str(e)[:120])

## Spark (profil `spark`)

Driver berjalan di container ini, executor di `spark-worker`; karena itu `spark.driver.host` harus nama service Jupyter. Versi pyspark klien harus sama dengan cluster (3.5.3).

In [ ]:
import os
try:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.appName('olist-v2-check').master(os.environ['SPARK_MASTER'])
             .config('spark.driver.host', 'jupyter').getOrCreate())
    print('OK Spark', spark.version, '| jumlah:', spark.range(1000).count())
except Exception as e:
    print('GAGAL Spark (profil spark menyala?):', type(e).__name__, str(e)[:160])

## Baca Parquet dari MinIO lewat S3A (setelah bronze/silver ada)

Lewati sel ini sampai ada data di bucket.

In [ ]:
# df = spark.read.parquet('s3a://silver/olist/orders/orders.parquet')
# df.printSchema(); df.show(5)

## Kontrak ODCS

Kontrak disimpan di `contracts/` (dibaca read-only oleh Airflow). Contoh membaca daftar tabelnya:

In [ ]:
import glob, yaml
for f in sorted(glob.glob('/home/jovyan/contracts/*.yaml')) or ['(belum ada kontrak di contracts/)']:
    print(f)